## Writing and Reading Time Series with TimeDB

This notebook shows how to write and read time series using three data formats:
1. **TimeSeries** — the default, via `timedatamodel`
2. **pandas** — `DataFrame` with `['valid_time', 'value']` columns
3. **numpy** — arrays in, arrays out

In [2]:
try:
    import google.colab
    import urllib.request
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/rebase-energy/timedb/main/examples/colab_setup.py',
        '/tmp/colab_setup.py'
    )
    exec(open('/tmp/colab_setup.py').read())
except ImportError:
    pass  # Not running in Google Colab


In [3]:
import timedb as td
from timedb import TimeSeries, Frequency
import pandas as pd
import numpy as np
from datetime import datetime, timezone, timedelta

### Setup

Create a fresh schema with wind power series across two sites.

In [4]:
td.delete()
td.create()

series_configs = [
    {'name': 'wind_power', 'unit': 'MW', 'labels': {'turbine': 'T01', 'site': 'Gotland', 'type': 'onshore'}},
    {'name': 'wind_power', 'unit': 'MW', 'labels': {'turbine': 'T02', 'site': 'Gotland', 'type': 'onshore'}},
    {'name': 'wind_power', 'unit': 'MW', 'labels': {'turbine': 'T03', 'site': 'Gotland', 'type': 'offshore'}},
    {'name': 'wind_power', 'unit': 'MW', 'labels': {'turbine': 'T01', 'site': 'Skane', 'type': 'onshore'}},
]

for config in series_configs:
    td.create_series(**config)

print(f"Created {len(series_configs)} series")

Creating database schema...
  Skipped (not supported): SELECT add_retention_policy('overlapping_short',  INTERVAL '...
  Skipped (not supported): SELECT add_retention_policy('overlapping_medium', INTERVAL '...
  Skipped (not supported): SELECT add_retention_policy('overlapping_long',   INTERVAL '...
  Skipped (not supported): ALTER TABLE flat SET (timescaledb.compress, timescaledb.comp...
  Skipped (not supported): SELECT add_compression_policy('flat', INTERVAL '14 days', if...
  Skipped (not supported): ALTER TABLE overlapping_short SET (timescaledb.compress, tim...
  Skipped (not supported): SELECT add_compression_policy('overlapping_short', INTERVAL ...
  Skipped (not supported): ALTER TABLE overlapping_medium SET (timescaledb.compress, ti...
  Skipped (not supported): SELECT add_compression_policy('overlapping_medium', INTERVAL...
  Skipped (not supported): ALTER TABLE overlapping_long SET (timescaledb.compress, time...
  Skipped (not supported): SELECT add_compression_policy('over

### With TimeSeries

`TimeSeries` is the default format for both writing and reading. Create one with `timedatamodel` and pass it directly to `insert()`. Calling `read()` returns a `TimeSeries` by default.

In [5]:
base_time = datetime(2025, 1, 1, 0, 0, tzinfo=timezone.utc)
dates = [base_time + timedelta(hours=i) for i in range(24)]

# Write
ts_in = TimeSeries(
    Frequency.PT1H,
    name="wind_power",
    unit="MW",
    timestamps=dates,
    values=[50.0 + i * 1.5 for i in range(24)],
)
td.get_series("wind_power").where(site="Gotland", turbine="T01").insert(ts_in)

# Read
ts_out = td.get_series("wind_power").where(site="Gotland", turbine="T01").read()
print(type(ts_out))
print(f"name={ts_out.name}, unit={ts_out.unit}, frequency={ts_out.frequency}")
print(f"{len(ts_out.values)} points, first={ts_out.values[0]}")

<class 'timedatamodel.timeseries.TimeSeries'>
name=wind_power, unit=MW, frequency=PT1H
24 points, first=50.0


### With pandas

Pass a `DataFrame` with columns `['valid_time', 'value']`. Use `read(output="pandas")` to get a DataFrame back.

In [6]:
# Write
df_in = pd.DataFrame({
    'valid_time': dates,
    'value': [55.0 + i * 1.2 for i in range(24)],
})
td.get_series("wind_power").where(site="Gotland", turbine="T02").insert(df_in)

# Read
df_out = td.get_series("wind_power").where(site="Gotland", turbine="T02").read(output="pandas")
print(type(df_out))
print(df_out.head())

<class 'pandas.core.frame.DataFrame'>
                           value
valid_time                      
2025-01-01 00:00:00+00:00   55.0
2025-01-01 01:00:00+00:00   56.2
2025-01-01 02:00:00+00:00   57.4
2025-01-01 03:00:00+00:00   58.6
2025-01-01 04:00:00+00:00   59.8


In [6]:
### With numpy

For writing, wrap numpy arrays in a `DataFrame`. Use `read(output="numpy")` to get arrays back.

In [7]:
# Write
timestamps = np.array(dates)
values = np.array([60.0 + i * 0.8 for i in range(24)])

df_np = pd.DataFrame({'valid_time': timestamps, 'value': values})
td.get_series("wind_power").where(site="Gotland", turbine="T03").insert(df_np)

# Read
arr = td.get_series("wind_power").where(site="Gotland", turbine="T03").read(output="numpy")
print(type(arr))
print(f"shape={arr.shape}")
print(arr[:5])

### Explore Series

Use `list_series()` to see full metadata and `list_labels()` for quick label discovery.

In [10]:
all_wind = td.get_series("wind_power")
print(f"All wind power series ({all_wind.count()}):\n")
for s in all_wind.list_series():
    print(f"  id={s['series_id']}  unit={s['unit']}  labels={s['labels']}")

print()
for site in all_wind.list_labels("site"):
    site_wind = all_wind.where(site=site)
    turbines = site_wind.list_labels("turbine")
    print(f"{site}: {site_wind.count()} turbines — {turbines}")

### Progressive Filtering

Start broad and narrow down with `.where()`. Each call returns a new collection.

In [8]:
wind = td.get_series("wind_power")
print(f"All wind power: {wind.count()} series")

gotland = wind.where(site="Gotland")
print(f"  Gotland: {gotland.count()} series")

onshore = gotland.where(type="onshore")
print(f"    Onshore: {onshore.count()} series")
print(f"    Turbines: {onshore.list_labels('turbine')}")

All wind power: 4 series
  Gotland: 3 series
    Onshore: 2 series
    Turbines: ['T01', 'T02']


### Summary

| Format | Write | Read |
|---|---|---|
| **TimeSeries** | `insert(ts)` | `read()` (default) |
| **pandas** | `insert(df)` | `read(output="pandas")` |
| **numpy** | `insert(df)` (wrap arrays in DataFrame) | `read(output="numpy")` |

Other output options: `output="polars"` for Polars DataFrames.